In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, HBox, HTML, Output, Layout
from IPython.display import display, clear_output

# ============================================================
# FLOATING-POINT QUANTIZER: SNR VS SIGNIFICAND BITS
# ============================================================

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:550px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:8px;
">
Floating-Point Quantization and SNR
</div>

<div style="margin-bottom:5px;">
Under the PQN approximation, the floating-point quantization-noise power is proportional to the input-signal power.
</div>

<div style="margin-bottom:5px;">
The resulting theoretical SNR is 7.44 + 6.02p dB, where p is the number of significand bits.
</div>

<div style="margin-bottom:5px;">
If the exponent-noise PQN assumption is relaxed, the SNR remains between two theoretical bounds.
</div>

<div>
<b>This notebook:</b> demonstrates the approximately 6.02 dB SNR improvement obtained from every additional significand bit.
</div>

</div>
""")

# ============================================================
# SLIDER
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='145px',
    min_width='145px'
)

p_slider = IntSlider(
    min=2,
    max=16,
    step=1,
    value=8,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# VALUE LABEL
# ============================================================

value_layout = Layout(
    width='55px',
    min_width='55px',
    margin='0px 0px 0px 4px'
)

p_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">8</div>',
    layout=value_layout
)

# ============================================================
# LABEL
# ============================================================

label_layout = Layout(
    width='155px',
    min_width='155px'
)

p_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Significand bits p:</div>',
    layout=label_layout
)

# ============================================================
# CONTROL ROW
# ============================================================

row_layout = Layout(
    width='390px',
    min_width='390px',
    height='38px',
    min_height='38px',
    align_items='center',
    overflow='visible'
)

p_row = HBox(
    [
        p_label,
        p_slider,
        p_value
    ],
    layout=row_layout
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#4f5f9b;
            margin-bottom:8px;
        ">
        SNR Parameter
        </div>
        """),

        p_row,

        HTML("""
        <div style="
            font-family:Arial;
            font-size:14px;
            line-height:1.40;
            margin-top:8px;
        ">
        Move <b>p</b> to select the number of bits used for the significand.
        The theoretical curves themselves do not change; the selected operating point moves along them.
        </div>
        """)
    ],
    layout=Layout(
        width='420px',
        min_width='420px',
        padding='10px 12px 12px 12px',
        border='1px solid #c3cae2',
        overflow='visible'
    )
)

# ============================================================
# TOP TWO-COLUMN LAYOUT
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1020px',
        align_items='flex-start',
        justify_content='space-between',
        gap='16px',
        margin='0px 0px 10px 0px',
        overflow='visible'
    )
)

# ============================================================
# OUTPUT AREAS
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

result_html = HTML()

# ============================================================
# SNR FUNCTIONS
# ============================================================

def snr_pqn(p):

    return 7.44 + 6.02 * p

def snr_lower(p):

    return 4.77 + 6.02 * p

def snr_upper(p):

    return 10.79 + 6.02 * p

# ============================================================
# MAIN PLOT FUNCTION
# ============================================================

def plot_snr(p_selected):

    # --------------------------------------------------------
    # FIXED BIT RANGE
    # --------------------------------------------------------

    p = np.arange(
        1,
        18
    )

    # --------------------------------------------------------
    # THEORETICAL CURVES
    # --------------------------------------------------------

    snr_center = snr_pqn(
        p
    )

    snr_low = snr_lower(
        p
    )

    snr_high = snr_upper(
        p
    )

    # --------------------------------------------------------
    # SELECTED VALUES
    # --------------------------------------------------------

    selected_center = snr_pqn(
        p_selected
    )

    selected_low = snr_lower(
        p_selected
    )

    selected_high = snr_upper(
        p_selected
    )

    # --------------------------------------------------------
    # NOISE-TO-SIGNAL POWER RATIO
    #
    # E{v²}/E{x²} = 0.180 * 2^(-2p)
    # --------------------------------------------------------

    noise_ratio = 0.180 * 2.0 ** (
        -2.0 * p
    )

    selected_noise_ratio = 0.180 * 2.0 ** (
        -2.0 * p_selected
    )

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.6, 6.9)
    )

    gs = fig.add_gridspec(
        1,
        2,
        wspace=0.30
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    # ========================================================
    # GRAPH 1
    # SNR VS SIGNIFICAND BITS
    # ========================================================

    ax1.fill_between(
        p,
        snr_low,
        snr_high,
        alpha=0.14,
        label='Theoretical range'
    )

    ax1.plot(
        p,
        snr_center,
        linewidth=2.2,
        label='PQN prediction'
    )

    ax1.plot(
        p,
        snr_low,
        linestyle='--',
        linewidth=1.4,
        label='Lower bound'
    )

    ax1.plot(
        p,
        snr_high,
        linestyle='--',
        linewidth=1.4,
        label='Upper bound'
    )

    ax1.axvline(
        p_selected,
        linestyle=':',
        linewidth=1.2
    )

    ax1.plot(
        p_selected,
        selected_center,
        marker='o',
        markersize=8
    )

    ax1.plot(
        p_selected,
        selected_low,
        marker='o',
        markersize=6
    )

    ax1.plot(
        p_selected,
        selected_high,
        marker='o',
        markersize=6
    )

    ax1.set_xlim(
        1,
        17
    )

    ax1.set_ylim(
        0,
        120
    )

    ax1.set_xticks(
        np.arange(
            2,
            18,
            2
        )
    )

    ax1.set_xlabel(
        'Significand bits p',
        fontsize=10
    )

    ax1.set_ylabel(
        'SNR (dB)',
        fontsize=10
    )

    ax1.set_title(
        'Floating-Point Quantizer SNR',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax1.legend(
        loc='upper left',
        fontsize=8
    )

    # ========================================================
    # GRAPH 2
    # NORMALIZED QUANTIZATION-NOISE POWER
    # ========================================================

    ax2.semilogy(
        p,
        noise_ratio,
        linewidth=2.2,
        marker='o',
        markersize=4,
        label='E{v²} / E{x²}'
    )

    ax2.axvline(
        p_selected,
        linestyle=':',
        linewidth=1.2
    )

    ax2.plot(
        p_selected,
        selected_noise_ratio,
        marker='o',
        markersize=8
    )

    ax2.set_xlim(
        1,
        17
    )

    ax2.set_ylim(
        1e-12,
        1e-1
    )

    ax2.set_xticks(
        np.arange(
            2,
            18,
            2
        )
    )

    ax2.set_xlabel(
        'Significand bits p',
        fontsize=10
    )

    ax2.set_ylabel(
        'Normalized noise power',
        fontsize=10
    )

    ax2.set_title(
        'Quantization-Noise Power',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        which='major',
        linestyle=':',
        alpha=0.4
    )

    ax2.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.91,
        bottom=0.11
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.55;
        width:980px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Selected significand length:</b>
    p = {p_selected} bits

    <br>

    <b>PQN prediction:</b>
    SNR = 7.44 + 6.02p = <b>{selected_center:.2f} dB</b>

    <br>

    <b>Theoretical bounds:</b>
    {selected_low:.2f} dB ≤ SNR ≤ {selected_high:.2f} dB

    <br>

    <b>Normalized quantization-noise power:</b>
    E{{v²}} / E{{x²}} = {selected_noise_ratio:.3e}

    <br>

    <b>Effect of one additional bit:</b>
    approximately +6.02 dB SNR and approximately 4× lower quantization-noise power.

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    p_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {p_slider.value}
    </div>
    """

    with graph_output:

        clear_output(
            wait=True
        )

        plot_snr(
            p_slider.value
        )

# ============================================================
# CONNECT CONTROL
# ============================================================

p_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #c8cee5;
    background:#f8f9fe;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
Under the PQN assumptions, the floating-point quantization-noise power is proportional to E{x²}, so the SNR depends primarily on the significand length p rather than on the absolute input-signal power.
</div>

<div style="margin-bottom:4px;">
Because E{v²}/E{x²} is proportional to 2<sup>−2p</sup>, every additional significand bit reduces the normalized noise power by approximately a factor of four.
</div>

<div>
Consequently, the SNR increases linearly by approximately <b>6.02 dB per additional bit</b>; the upper and lower lines indicate the theoretical range obtained when the exponent-noise PQN assumption is not imposed.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='visible'
    )
)

display(main_layout)